In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json, math
text = open("../data/splits/train.txt", encoding="utf-8").read()
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text])
print(len(chars), len(data))
val_text = open("../data/splits/val.txt", encoding="utf-8").read()
val_data = torch.tensor([stoi[c] for c in val_text if c in stoi])
print(len(val_text), len(val_data))

352 9506558
787771 787768


In [5]:
def get_batch(data, block, batch_size):
    s = torch.randint(0, len(data) - block - 1, (batch_size,))
    x = torch.stack([data[i : i + block] for i in s])
    y = torch.stack([data[i + block] for i in s])
    return x, y
torch.manual_seed(1337)
x, y = get_batch(data, 3, 32)
print(x.shape, y.shape)
print(repr("".join(itos[int(t)] for t in x[0])), "->", repr(itos[int(y[0])]))

torch.Size([32, 3]) torch.Size([32])
'ĩto' -> 'n'


In [6]:
emb = nn.Embedding(len(chars), 10)
e = emb(x)
print(e.shape)
flat = e.view(32, -1)
print(flat.shape)

torch.Size([32, 3, 10])
torch.Size([32, 30])


In [22]:
class MLP(nn.Module):
    def __init__(self, vocab_size, block, emb_dim, hidden):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim)
        self.hidden = nn.Linear(block * emb_dim, hidden)
        self.out = nn.Linear(hidden, vocab_size)

    def forward(self, x):
        e = self.emb(x)
        e = e.view(e.shape[0], -1)
        h = torch.tanh(self.hidden(e))
        return self.out(h)
BLOCK = 3
torch.manual_seed(1337)
model = MLP(len(chars), BLOCK, 20, 200)
print(sum(p.numel() for p in model.parameters()))


89992


In [23]:
@torch.no_grad()
def val_loss_sample(model, n=20):
    model.eval()
    losses = []
    for _ in range(n):
        xb, yb = get_batch(val_data, BLOCK, 512)
        losses.append(F.cross_entropy(model(xb), yb).item())
    model.train()
    return sum(losses) / len(losses)

opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
steps = 20000
for step in range(steps):
    if step == steps // 2:
        for g in opt.param_groups:
            g["lr"] = 3e-4
    xb, yb = get_batch(data, BLOCK, 64)
    loss = F.cross_entropy(model(xb), yb)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 2000 == 0 or step == steps - 1:
        print(step, round(loss.item(), 4), round(val_loss_sample(model), 4))


0 5.7964 5.6978
2000 1.928 2.1042
4000 1.847 2.0477
6000 2.4404 2.0642
8000 1.4487 2.0112
10000 2.0931 1.9901
12000 2.0351 1.9283
14000 1.7035 1.932
16000 1.9951 1.8859
18000 1.8473 1.8804
19999 1.9237 1.9134


In [24]:
@torch.no_grad()
def full_val_bpc(model, val_data, block):
    model.eval()
    X = val_data.unfold(0, block, 1)[:-1]
    Y = val_data[block:]
    total = 0.0
    for i in range(0, len(Y), 65536):
        logits = model(X[i : i + 65536])
        total += F.cross_entropy(logits, Y[i : i + 65536], reduction="sum").item()
    model.train()
    nats = total / len(Y)
    return nats / math.log(2)

bpc = full_val_bpc(model, val_data, BLOCK)
print("val bpc", round(bpc, 3))


val bpc 2.728


In [13]:
@torch.no_grad()
def generate(model, n_chars, seed):
    g = torch.Generator().manual_seed(seed)
    context = [stoi["\n"]] * BLOCK
    out = []
    for _ in range(n_chars):
        x = torch.tensor([context])
        probs = F.softmax(model(x), dim=-1)
        nxt = torch.multinomial(probs, 1, generator=g).item()
        out.append(itos[nxt])
        context = context[1:] + [nxt]
    return "".join(out)

model.eval()
print(generate(model, 300, 0))

json.dump({"model": "mlp", "context": BLOCK, "emb_dim": 10, "hidden": 200,
"steps": steps, "val_bpc": round(bpc, 3)}, open("../results/mlp.json", "w"),
indent=2)

Yte3 mbĩ we Ndaĩ ata nete kwa."
23. * YA Kuma ndekeanaa kwasũnĩ, kũtwatyene nokia kw'yu." 13. Amu. yĩkatia ithewe atũmwe ũka, kũũũmwendeee Yesva mũkw'a na Aminyanĩsya asoa.

Kwĩ nĩkw'ĩ na tũtũmavine nzya kũthĩweeya.

Ndũumĩtĩkĩĩoo tony'o; ũmũno wa wĩa mũambĩ 'undi kwa, ũndũ atangoo, nĩwetikuwa ithuk


| Run | Change | Val bpc |
|---|---|---|
| Baseline | context 3, emb 10, hidden 200 | 2.794 |
| A | context 5 | 2.662 |
| B | hidden 400 | 2.801 |
| C | emb 20 | 2.728 |